# 02 · Categories and representations

**PRT2030 · Dataset inspection · Study notebook**

Notebook 01 checked the structure, missing cells and exact repeated rows. Now we inspect the values inside
populated cells. A value may be present but uninformative; equivalent labels may differ in formatting;
and a date-looking string may fail to represent a usable date.

Our sequence is **categories → useful information → text comparison → row matching → number/date conversion → evidence and log**.

**How to use this study resource:** Read the inspection question, follow the comments as you run the code,
and explain what the output tells you. Run from the top; later checks reuse named results from earlier cells.
Each notebook loads its own data and runs independently. Use your existing `wildfires.csv` beside the notebooks.

**One source row represents a reported wildfire record.** Matching names or rows do not establish that records
describe the same physical fire: this extract has no incident identifier. Fire sizes are in acres.
We inspect temporary results and evidence tables while keeping `raw` unchanged.

The worked findings refer to the supplied teaching extract (1,880,465 records and eight columns).
They are reference figures from the supplied notebooks, not results of a new run in this revision.
Code outputs have been cleared so your next top-to-bottom run produces a consistent set of results.
If your file differs, use your own computed results. The teaching target remains pandas 3.0.5.
The worked log wording describes the reference findings; revise the written judgements if your results differ.

Function and method names in the documentation lines are clickable links to their specific official reference pages.

**Reading the examples:** The small values introduced as examples explain the logic;
they are not new measurements from the wildfire file. Comments explain the purpose,
intermediate results and reasons for the choices. Apply the stated prerequisites
before interpreting a zero flag count as a pass.


## 1. Load the data

Start a fresh inspection of the original extract. The variables used here are recreated within this notebook;
you do not need Notebook 01's kernel or variables.

**Documentation:** [pd.read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [len()](https://docs.python.org/3.14/library/functions.html#len).


In [ ]:
# We need a trustworthy starting table before we can judge its values. Keep this import as
# raw so every finding can be traced back to the same original records.

# Pandas gives us labelled tables and columns. The short name pd lets us use that library
# consistently in all later cells.
import pandas as pd

# The CSV becomes a DataFrame: one row is one reported fire, and each column supplies a
# different fact about it. Loading it afresh lets this notebook run independently of the
# others.
raw = pd.read_csv("wildfires.csv")

# A missing-value check asks about records, so its whole-file denominator must be the row
# count. Counting all cells would make the affected percentage artificially small.
total_records = len(raw)
# Stop on an empty import: there is no population to inspect and division by the row count
# would be undefined. An empty file cannot demonstrate that its values passed a check.
if total_records == 0:
    raise ValueError("The dataset contains no records. Check the import before continuing.")

# Record the environment alongside the count so an unexpected result can be compared with
# another run. A mixed-type warning is a clue to inspect a column; it does not identify which
# values are wrong.
print("pandas version:", pd.__version__)
print("Reported wildfire records:", total_records)


## 2. Examine the categories

We want to compare records by cause, state and size class. Before treating these as reliable groups,
look at the labels that actually occur and how often each appears.


### 2.1 Read the fire cause labels

The cause field is populated in the teaching extract, but we have not yet established what those labels tell us.

**Inspection question:** Which cause labels occur, and how many records use each?

**Documentation:** [Series.value_counts()](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html).


In [ ]:
# Before comparing fires by cause, see which labels would actually become groups in that
# analysis.

# The result has cause LABELS as its index and occurrence counts as its values; it no longer
# has one row per fire. Including missing values prevents us from silently omitting part of
# the source population.
cause_counts = raw["fire_cause"].value_counts(dropna=False)
# A populated label such as Missing/Undefined will appear as an ordinary group. Its frequency
# matters because it contains records for which we cannot identify a known cause.
display(cause_counts)


### 2.2 Distinct values and proportions

Counts show the frequency of each label. Distinct-value checks show the available choices, while percentages put their frequencies into context.

**Inspection question:** How many cause labels are used, and what share of all records does each represent?

**Documentation:** [Series.unique()](https://pandas.pydata.org/docs/reference/api/pandas.Series.unique.html) · [Series.nunique()](https://pandas.pydata.org/docs/reference/api/pandas.Series.nunique.html) · [Series.round()](https://pandas.pydata.org/docs/reference/api/pandas.Series.round.html).


In [ ]:
# Use the label list, number of choices and their proportions to ask different questions
# about the same categorical field.

# The list reveals spelling and meaning, while the distinct count shows how many present
# label choices exist. A count of 13 cannot tell us whether one of those labels is
# unexpected.
print("Observed cause labels:", raw["fire_cause"].unique())
print("Number of distinct present labels:", raw["fire_cause"].nunique())

# Each label count is divided by all records, so the result answers "What share of the whole
# file has this label?" For 25 of 100 records the result is 25%, including any missing-label
# group retained above.
cause_percentages = cause_counts / total_records * 100
print("Cause label -> percentage of all records:")
display(cause_percentages.round(2))

# Repeat the overview for the other grouping fields. Observed choices describe the file; they
# cannot validate themselves. A separate documented list is needed to decide whether a label
# is allowed.
print("State/jurisdiction frequencies:")
display(raw["state"].value_counts(dropna=False))
print("Size-class frequencies:")
display(raw["fire_class"].value_counts(dropna=False))


## 3. Distinguish missing cells from undefined information

We have seen the actual labels. Now ask whether a filled-in cell supplies the information our analysis needs.
First examine the explicit cause label `Missing/Undefined`; then prepare name values for similar text checks.


### 3.1 A populated cell can still lack useful information — C01

A cause label of Missing/Undefined is ordinary text, so a missing-cell check will not flag it. For cause analysis, it still leaves the cause unknown.

**Inspection question:** How many records lack a defined cause even though their cause cell is populated?

**Documentation:** [Series.isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [Series.sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) · [round()](https://docs.python.org/3.14/library/functions.html#round).


In [ ]:
# A filled cell and a known cause are different requirements, so we must measure them with
# different conditions.

# isna asks whether information is absent from the cell. The equality test asks whether the
# present value explicitly says Missing/Undefined. That text is not a missing marker
# recognised by isna.
missing_cause = raw["fire_cause"].isna()
undefined_cause = raw["fire_cause"] == "Missing/Undefined"

# Keep the two counts separate: they describe different observations and may require
# different treatments. A source can legitimately allow an undefined label while still
# leaving the cause question unanswered.
print("Missing cause cells:", missing_cause.sum())
print("Missing/Undefined labels:", undefined_cause.sum())
# Use all records as the denominator because the question concerns the proportion of the
# whole file with this explicit label. A known-causes-only analysis would have a different
# population.
print("Undefined cause (% of all records):",
      round(undefined_cause.sum() / total_records * 100, 2))


### 3.2 Prepare fire-name values for comparison

To find blank names, possible placeholders and formatting variations, we need comparable text. We will keep the original name, a version without surrounding whitespace, and a version for case-insensitive comparisons. Each Series retains the original row labels.

**Inspection question:** How can we compare names while preserving the original spelling as evidence?

**Documentation:** [Series.astype()](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) · [Series.str.strip()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html) · [Series.str.casefold()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.casefold.html) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).


In [ ]:
# We need several views of each name: the original as evidence, an outside-whitespace
# comparison, and a case-insensitive comparison.

# Use nullable text so absent names remain absent. Turning a missing value into literal text
# such as "nan" would invent a name and distort both missingness and distinct-label counts.
names = raw["fire_name"].astype("string")

# If " Pine Ridge " becomes "Pine Ridge", the difference reveals surrounding whitespace.
# Internal spacing stays intact, so "Pine  Ridge" is still different from "Pine Ridge".
trimmed_names = names.str.strip()

# Casefold lets "PINE RIDGE" and "Pine Ridge" share a comparison value. It is intended for
# matching, not choosing how a name should be displayed; it also handles some case
# differences beyond English letters.
comparison_names = trimmed_names.str.casefold()

# The three Series retain the same row labels, so each preview row compares versions of the
# same source name. Matching by labels prevents us from accidentally comparing one fire with
# another.
name_preparation_preview = pd.DataFrame({
    "Original": names.head(5),
    "Trimmed": trimmed_names.head(5),
    "Comparison": comparison_names.head(5)
})
# The first five names need not change. This preview explains the preparation; the later
# filters find records where it actually makes a difference.
display(name_preparation_preview)


### 3.3 Blank strings and possible placeholders — T02

Now use the prepared names to look for filled-in values that may provide little information. A whitespace-only string becomes empty after trimming; a word such as unknown may be a placeholder.

**Inspection question:** Do present name values contain only whitespace or possible placeholder words?

**Documentation:** [Series.fillna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html) · [Series.isin()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) · [Series.value_counts()](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html).


In [ ]:
# Some present names supply no useful label. Screen blanks and candidate placeholders without
# assuming they mean the same thing as a missing cell.

# Three spaces become an empty string after trimming and match this blank condition. An
# absent name yields an unknown text comparison; fillna(False) keeps it outside this separate
# blank-string screen, with M01 still recording its absence.
blank_names = (trimmed_names == "").fillna(False)

# Choose candidate words explicitly so the screen is reproducible. "unknown" suggests
# unavailable information, but a source might use it as an actual recorded name; the match
# therefore requests investigation.
placeholder_words = ["unknown", "unnamed", "not named", "none"]

# The case-insensitive, trimmed comparison means " UNKNOWN " matches the same candidate as
# "unknown". Membership tests look for complete listed values, not the word inside a longer
# name.
possible_placeholders = comparison_names.isin(placeholder_words)
print("Whitespace-only names:", blank_names.sum())
print("Possible placeholder names:", possible_placeholders.sum())

# Count the original spellings to show what was really supplied. The comparison value finds
# candidates; the original value and source context support the later decision.
display(names.loc[possible_placeholders].value_counts())


## 4. Examine spaces and letter case — T01

We have checked whether a name supplies information. Next ask whether inconsistent formatting can split
equivalent recorded labels into separate groups. Sharing a label does not prove that records describe the same fire.


### 4.1 Detect surrounding whitespace

Reuse names and trimmed_names from section 3.2. If a present original name differs from its trimmed version, surrounding whitespace was removed.

**Inspection question:** Which present names contain whitespace at the beginning or end?

**Documentation:** [Series.notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [Series.fillna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html) · [Series.sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html).

**Whitespace rule example:** Compare a present original name with its trimmed version.
`" Oak "` differs from `"Oak"` and is flagged; `"Oak"` is unchanged and is not flagged.
A missing name has no spelling to compare and remains outside the checked population.
The flag concerns recorded text, not whether two records describe the same physical fire.


In [ ]:
# We can identify surrounding whitespace by asking whether a present name changes when we
# remove whitespace from its ends.

# An absent name has no spelling to inspect. Keep eligibility separate so the whitespace
# count and later denominator describe only names that could be compared.
present_names = names.notna()
# Compare each original with its own trimmed version. "Oak" stays equal and gives False; "
# Oak " differs and gives True.
changed_by_trimming = names != trimmed_names

# & combines element-by-element decisions: both "name exists" and "trimming changed it" must
# be true for this source row. Parent conditions retain their row labels, so the correct
# decisions line up.
name_has_spaces = present_names & changed_by_trimming

# An unknown comparison cannot demonstrate whitespace. Filling unknown flags with False makes
# the selection usable; it does not repair or certify a missing name.
name_has_spaces = name_has_spaces.fillna(False)

# Sum the full-column decisions before any preview is taken. This gives affected source
# records, not the number of distinct spellings.
space_count = name_has_spaces.sum()
print("Names with surrounding whitespace:", space_count)


### 4.2 Make the whitespace visible

A table can hide leading and trailing spaces visually. Put each original and trimmed name together, then show quoted representations so the differences can be seen.

**Inspection question:** What changed in some of the flagged names?

**Documentation:** [Series.loc](https://pandas.pydata.org/docs/reference/api/pandas.Series.loc.html) · [Series.map()](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html) · [repr()](https://docs.python.org/3.14/library/functions.html#repr).


In [ ]:
# Ordinary table formatting can hide outside spaces, so make the same affected names visible
# in two representations.

space_examples = pd.DataFrame()
# Start with eight flagged originals. This assignment gives the preview their source labels;
# it does not renumber them as 0 through 7.
space_examples["Original"] = names.loc[name_has_spaces].head(8)

# Retrieve trimmed names using those exact labels. If the chosen source rows are 17 and 42,
# we need trimmed rows 17 and 42, not the first two trimmed values.
space_examples["After trimming"] = trimmed_names.loc[space_examples.index]

# repr adds quotes and visible escape sequences: " Oak " has spaces inside its quotes, while
# "Oak" does not; a tab can appear as \t. This helps us see why the equality test flagged the
# value.
space_examples["Original shown with quotes"] = space_examples["Original"].map(repr)
space_examples["Trimmed shown with quotes"] = space_examples["After trimming"].map(repr)
# The eight examples illustrate the difference. Keep space_count as the full affected count;
# the preview has intentionally omitted other matching records.
display(space_examples)


### 4.3 Measure the effect on distinct names

Finding whitespace does not tell us how many name groups it splits. Compare distinct label counts before and after each temporary transformation.

**Inspection question:** Do formatting changes cause previously different labels to match?

**Documentation:** [Series.nunique()](https://pandas.pydata.org/docs/reference/api/pandas.Series.nunique.html).


In [ ]:
# A whitespace count measures affected records; distinct counts instead reveal whether
# formatting has split labels into extra groups.

# Compare the number of observed label choices at each stage. If "Oak" and " Oak " are the
# only originals, two choices become one after trimming.
print("Distinct original names:", names.nunique())
print("Distinct after trimming:", trimmed_names.nunique())
# A further reduction would show labels distinguished only by case after whitespace is
# removed. No reduction here means casefold found no extra merged labels in this file; it
# does not make the method unnecessary for other files.
# A merged label is not a merged incident: different fires may both be called Oak. These
# counts describe group labels, not how many real fires exist.
print("Distinct after trimming and case comparison:", comparison_names.nunique())


### 4.4a Practise finding alternative versions of a name

A lower distinct-name count suggests merged groups, but we need to see their original spellings. This small teaching example shows why removing duplicate pairs must come before testing the comparison column.

**Inspection question:** How can we distinguish repeated identical names from different versions of a name?

**Documentation:** [Series.str.strip()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html) · [Series.str.casefold()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.casefold.html) · [DataFrame.drop_duplicates()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) · [Series.duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html).


In [ ]:
# The key question is whether DIFFERENT original spellings share one comparison value;
# frequent repetition of a single spelling should not manufacture a variant.

# There are two identical "Pine Ridge" values plus two different spellings of that name. Oak
# has only one spelling, so it should not appear in the final variant table.
demo_names = pd.Series(["Pine Ridge", "Pine Ridge", " PINE RIDGE ", "pine ridge", "Oak"],
                       dtype="string")
# Trimming and casefolding make the three distinct Pine Ridge spellings comparable. The
# parallel Series keep each original attached to its own comparison value.
demo_comparison = demo_names.str.strip().str.casefold()
demo_versions = pd.DataFrame({"Original": demo_names, "Comparison": demo_comparison})

# Remove duplicates across BOTH columns first. The repeated identical pair contributes one
# row, leaving three genuinely different originals with comparison value "pine ridge".
demo_versions = demo_versions.drop_duplicates()

# Now compare only the comparison column. Repetition here means at least two distinct
# originals survived for that value. keep=False includes all spellings in the group,
# including its first row.
demo_several_versions = demo_versions["Comparison"].duplicated(keep=False)
demo_matches = demo_versions.loc[demo_several_versions].copy()

# Quote the originals so the leading and trailing spaces remain visible. The expected three
# rows show spelling variants; Oak is excluded because its comparison group has only one
# original.
demo_matches["Original shown with quotes"] = demo_matches["Original"].map(repr)
display(demo_matches)


### 4.4b Find alternative versions in the wildfire names

Apply the demonstrated sequence to names and comparison_names from section 3.2. The final table will contain one row per distinct original/comparison pair, rather than every fire record.

**Inspection question:** Which distinct original spellings share a comparison value?

**Documentation:** [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [DataFrame.dropna()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) · [DataFrame.drop_duplicates()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) · [Series.duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html) · [DataFrame.sort_values()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html).


In [ ]:
# Apply the small demonstration to the source: identify shared comparison values among
# distinct original spellings, rather than repeated occurrences of one spelling.

# Pair each original with its own comparison value using their shared source labels. Before
# filtering, each row still corresponds to a fire record.
name_versions = pd.DataFrame({"Original": names, "Comparison": comparison_names})

# An absent original cannot demonstrate a spelling variation. Remove these comparison rows
# only; missing names remain in raw and in their missingness finding.
name_versions = name_versions.dropna()

# Collapse identical PAIRS: fifty occurrences of "Oak" now contribute one pair, so frequency
# cannot be mistaken for fifty variants. After this step, one row means one distinct
# original/comparison pair.
name_versions = name_versions.drop_duplicates()

# If "Oak" and " OAK " both become "oak", that comparison value repeats among distinct pairs.
# keep=False marks both originals; checking duplicates before collapsing pairs would also
# flag a single spelling repeated many times.
several_versions = name_versions["Comparison"].duplicated(keep=False)

# Keep only the distinct pairs whose shared comparison value has multiple originals. A lone
# Oak spelling is removed from this result because it supplies no evidence of a variant,
# regardless of how often it occurs in raw.
name_versions = name_versions.loc[several_versions]
# Put each comparison group together so the shared value and different originals can be read
# side by side. Sorting changes the view order, not source identity.
name_versions = name_versions.sort_values("Comparison")

# A small display may end part-way through a group. Keep the complete name_versions table for
# reviewing all partners, and remember that its row count is distinct spellings, not affected
# source records.
name_versions_preview = name_versions.head(10).copy()
name_versions_preview["Original shown with quotes"] = name_versions_preview["Original"].map(repr)
display(name_versions_preview)


### 4.5 Apply the same text check to other fields

We understand the comparison for one field. A short loop can now repeat those familiar operations for the other categorical fields.

**Inspection question:** Do state, size-class or cause labels show the same formatting concerns?

**Documentation:** [Series.astype()](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) · [Series.str.strip()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html) · [Series.str.casefold()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.casefold.html).


In [ ]:
# Once the one-field logic is understood, reuse it to ask the same formatting question for
# state, class and cause labels.

# The loop repeats the same steps for one field at a time; column selects which source Series
# supplies this iteration. This prevents slightly different hand-written checks for otherwise
# comparable fields.
for column in ["state", "fire_class", "fire_cause"]:
    values = raw[column].astype("string")
    trimmed = values.str.strip()
    folded = trimmed.str.casefold()
    # A present value must differ after trimming to be flagged. Compare distinct counts as
    # well: a change in many records need not create the same number of extra label groups.
    spaces = (values.notna() & (values != trimmed)).fillna(False)
    print(column, "surrounding-whitespace flags:", spaces.sum())
    # The three counts distinguish changes in whitespace from further merges under case
    # comparison. Temporary variables are reused next iteration; all source columns remain
    # available unchanged.
    print("Distinct: original / trimmed / case comparison:",
          values.nunique(), trimmed.nunique(), folded.nunique())


## 5. Find repeated rows hidden by formatting — R02

We have compared individual text labels. Now extend the idea to complete rows: records that fail an exact
comparison may match after selected text fields are compared consistently. All eight columns still participate.


### 5.1 Create a separate comparison DataFrame

To compare complete records, we need a whole table containing the temporary text versions alongside the unchanged numerical and date fields.

**Inspection question:** What would row comparisons find if selected text fields ignored surrounding whitespace and letter case?

**Documentation:** [DataFrame.copy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [Series.str.casefold()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.casefold.html).


In [ ]:
# To ask whether text formatting hides full-row matches, build a comparison table containing
# the same records but comparable text.

# Work on a separate table so the original spellings remain evidence. The copy retains every
# row label and every field; we are testing equivalence, not deleting records.
comparison = raw.copy()

# Only these four label fields use the trim-and-casefold comparison. Numerical values and
# date text still have to match as originally supplied, so the later duplicate test continues
# to compare complete rows.
for column in ["fire_name", "state", "fire_class", "fire_cause"]:
    text = raw[column].astype("string")
    text = text.str.strip()
    # Putting the comparison Series back into the copied table matches values by row label. A
    # name shared by two rows is insufficient: every other field must also match for the rows
    # to be flagged.
    comparison[column] = text.str.casefold()


### 5.2 Count and inspect newly flagged records

Compare group membership before and after the temporary formatting changes. A newly flagged record was not previously a member of an exact repeated group.

**Inspection question:** Which additional records now belong to matching groups?

**Documentation:** [DataFrame.duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) · [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [Series.map()](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html) · [repr()](https://docs.python.org/3.14/library/functions.html#repr).


In [ ]:
# Measure the additional review candidates introduced by our text comparison, keeping group
# membership separate from later-copy counts.

# Both membership flags include all partners in a repeated group. Using keep=False on both
# tables makes the before-and-after question comparable.
exact_members = raw.duplicated(keep=False)
comparison_members = comparison.duplicated(keep=False)

# ~exact_members means the record was NOT already in an exact repeated group. & keeps only
# records that now match partners under comparison and were not previously flagged.
new_members = comparison_members & ~exact_members
print("Original repeated-group members:", exact_members.sum())
print("Comparison repeated-group members:", comparison_members.sum())
print("Newly flagged records:", new_members.sum())

# Later-copy counts answer another question. If two unique rows become one matching pair,
# there are two newly flagged members but only one additional later copy.
later_before = raw.duplicated().sum()
later_after = comparison.duplicated().sum()
print("Additional later repetitions:", later_after - later_before)

# Show original values selected by the new flags so the formatting difference can be
# reviewed. Newly flagged rows alone may omit an already-flagged partner; a treatment
# decision needs the full matching group and source identity.
new_match_examples = raw.loc[new_members].head(10).copy()
new_match_examples["Name shown with quotes"] = new_match_examples["fire_name"].map(repr)
display(new_match_examples)


## 6. Inspect unusual characters — T03

Whitespace and case are only two text concerns. Invisible control characters may interfere with display; non-ASCII characters may be valid names or deserve an encoding review. The patterns are supplied inspection tools.

**Inspection question:** Which names contain control characters or characters outside basic ASCII?

**Documentation:** [Series.str.contains()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html) · [Series.drop_duplicates()](https://pandas.pydata.org/docs/reference/api/pandas.Series.drop_duplicates.html) · [repr()](https://docs.python.org/3.14/library/functions.html#repr).


In [ ]:
# A name can contain invisible controls or legitimate non-ASCII letters; use separate screens
# because their meanings differ.

# The pattern looks for any ASCII control such as a tab or newline. The raw-string prefix
# keeps the backslashes for the pattern; na=False leaves absent names out because they have
# no characters to test.
control_characters = names.str.contains(r"[\x00-\x1f\x7f]", na=False)

# The ^ inside the brackets means a character outside the basic ASCII range. An accented
# letter in "José" matches this screen but can be a completely correct part of a name.
non_ascii = names.str.contains(r"[^\x00-\x7f]", na=False)
print("Control-character records:", control_characters.sum())
print("Non-ASCII records:", non_ascii.sum())

# Collapse repeated spellings only for the preview, then quote them to expose invisible
# characters. The earlier flag sums count records; this shortened display counts neither all
# records nor all character occurrences.
display(names.loc[non_ascii].drop_duplicates().head(10).map(repr))


## 7. Test numeric representations

Text checks have revealed how labels are written. We next ask whether measurement and year values can be
interpreted numerically. Successful conversion establishes a usable representation; ranges and meaning follow in Notebook 03.


### 7.1 Practise on four small values

Use a small example to distinguish a convertible number, unconvertible text, a pre-existing absence and infinity.

**Inspection question:** What does numeric conversion produce for each kind of input?

**Documentation:** [Series()](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [pd.to_numeric()](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html).


In [ ]:
# Conversion is a representation test. Use four inputs to see why a missing result and an
# infinite result need different interpretations.

# "12.5" describes a number; "unknown" cannot supply a numeric measurement; None was absent
# before conversion; "inf" describes infinity.
example_text = pd.Series(["12.5", "unknown", None, "inf"])

# errors="coerce" keeps the check running by turning unconvertible text into a missing
# numeric result. The absent input also remains missing, while "inf" converts successfully
# despite not being a finite measurement.
example_numbers = pd.to_numeric(example_text, errors="coerce")
# Compare original and result side by side. Looking only at missing numeric results would mix
# conversion failures with pre-existing absences, so the source-presence condition is needed
# next.
display(pd.DataFrame({"Original": example_text, "Numeric result": example_numbers}))


### 7.2 Check fire size and year — P01

Apply the same conversion to the real fields. Count originally missing values separately from present values that fail conversion, and check infinity explicitly.

**Inspection question:** Can the present sizes and years be interpreted as finite numbers?

**Documentation:** [pd.to_numeric()](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) · [Series.notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [Series.isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [np.isinf()](https://numpy.org/doc/stable/reference/generated/numpy.isinf.html).


In [ ]:
# Before range or relationship checks, establish which source values can be read as numbers
# and distinguish why others cannot be used.

# Use a temporary numeric Series so the source representation remains available.
# Unconvertible present text becomes missing in this result and can therefore be located
# without stopping at the first failure.
size_numbers = pd.to_numeric(raw["fire_size"], errors="coerce")

# A failure needs BOTH conditions: the source was present and the numeric result is missing.
# An originally absent size is a completeness finding, not a conversion failure.
size_conversion_failed = raw["fire_size"].notna() & size_numbers.isna()
print("Originally missing sizes:", raw["fire_size"].isna().sum())
print("Present sizes checked:", raw["fire_size"].notna().sum())
print("Size conversion failures:", size_conversion_failed.sum())
# Retrieve the original failing values to explain what could not be read. A zero-length
# preview supports a zero result only for present inputs; missing inputs were outside this
# conversion test.
display(raw.loc[size_conversion_failed, ["fire_size"]].head(10))

import numpy as np
# Infinity can survive numeric conversion, so a missing-result test alone would miss it.
# Explicitly flag positive and negative infinity before ordinary size calculations.
infinite_sizes = np.isinf(size_numbers)
print("Infinite sizes:", infinite_sizes.sum())

# Repeat the representation check for year. Successful finite conversion is only the entry
# point: 1998.5 is readable as a number but still unsuitable as a whole reported year.
year_numbers = pd.to_numeric(raw["fire_year"], errors="coerce")
year_conversion_failed = raw["fire_year"].notna() & year_numbers.isna()
print("Originally missing years:", raw["fire_year"].isna().sum())
print("Year conversion failures:", year_conversion_failed.sum())
print("Infinite years:", np.isinf(year_numbers).sum())


## 8. Test dates and their text format — P02

Duration calculations need usable dates. We already know some dates are absent; now check whether the present
values can be interpreted using the extract's timestamp format. Keep absence, conversion failure and formatting differences distinct.


### 8.1a Distinguish absent dates from failed conversions

This short teaching example makes the failure rule visible before it is applied to the full dataset.

**Inspection question:** If two converted results are missing, how can we tell which one is a conversion failure?

**Documentation:** [pd.to_datetime()](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) · [Series.notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [Series.isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html).

**Conversion-failure rule examples:**

| Original input | Converted result | Interpretation |
| --- | --- | --- |
| `2001-07-19 00:00:00` | Usable date | Present input converted successfully. |
| `2001-99-19 00:00:00` | Missing datetime (`NaT`) | Present input failed: month 99 is impossible. |
| Originally missing | Missing datetime (`NaT`) | Absence, not a failed conversion of present text. |

The failure condition needs **present before conversion AND missing afterward**. Looking only
at the final missing result would mix the last two cases.


In [ ]:
# A date conversion can yield two missing results for two different reasons. Compare the
# originals to distinguish them.

# The first string is a real timestamp, month 99 makes the second impossible, and the third
# input was already absent.
demo_date_text = pd.Series(["2001-07-19 00:00:00", "2001-99-19 00:00:00", None])

# The format tells the converter which part is year, month, day and time. Coercion represents
# an unreadable or absent date as NaT, allowing us to inspect all inputs rather than stop at
# the first impossible one.
demo_dates = pd.to_datetime(demo_date_text, format="%Y-%m-%d %H:%M:%S", errors="coerce")

# Only the impossible timestamp was present and then became missing. The originally absent
# input cannot fail a conversion of present values, even though its final result is also NaT.
demo_failed = demo_date_text.notna() & demo_dates.isna()
# The three-column report makes that distinction visible: one valid conversion, one failure
# and one absence. We need the same distinction when recording real-date checks.
display(pd.DataFrame({"Original": demo_date_text, "Converted": demo_dates,
                      "Present value failed": demo_failed}))


### 8.1b Interpret the two wildfire date columns

Use the same rule to check containment and discovery values. Missing containment dates remain a completeness limitation, even if every present date converts successfully.

**Inspection question:** Which present timestamps fail conversion, and what date range can be read?

**Documentation:** [pd.to_datetime()](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) · [Series.min()](https://pandas.pydata.org/docs/reference/api/pandas.Series.min.html) · [Series.max()](https://pandas.pydata.org/docs/reference/api/pandas.Series.max.html).

**Applying the rule to real fields:** If an original containment value is `2001-99-19 00:00:00`,
it is a present input that fails the stated calendar conversion. If containment was originally
absent, it belongs to M02 instead. If every **present** value converts, that establishes no
conversion failures in the checked population; it does not make the absent endpoints available.


In [ ]:
# Prepare usable dates for later calculations while measuring present inputs that cannot be
# interpreted under the extract format.

# Keep the converted date Series separate from the source text. Its original row labels will
# let date comparisons and evidence selections refer to the same record.
contained_dates = pd.to_datetime(
    raw["contained_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
# Require an original value and an unavailable converted date. Missing containment dates
# already have M02; counting them again as conversion failures would confuse two different
# findings.
containment_parse_failed = raw["contained_date"].notna() & contained_dates.isna()
print("Originally missing containment dates:", raw["contained_date"].isna().sum())
# The checked population is present containment values, not all source records. A pass on
# that population cannot make the absent dates usable for duration.
print("Present containment dates checked:", raw["contained_date"].notna().sum())
print("Containment conversion failures:", containment_parse_failed.sum())
display(raw.loc[containment_parse_failed, ["contained_date"]].head(10))

# Apply the same format and failure logic to discovery dates so both endpoints can later be
# compared as dates.
discovery_dates = pd.to_datetime(
    raw["discovery_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
discovery_parse_failed = raw["discovery_date"].notna() & discovery_dates.isna()
print("Discovery conversion failures:", discovery_parse_failed.sum())

# Earliest and latest usable dates help reveal suspicious coverage or extreme dates.
# Readability alone does not establish date order, range plausibility or accuracy.
print("Discovery range:", discovery_dates.min(), "to", discovery_dates.max())
print("Containment range:", contained_dates.min(), "to", contained_dates.max())


### 8.2 Check the text shape separately

A timestamp can have the expected digit-and-separator pattern yet describe an impossible date, as the teaching example showed. Shape and calendar conversion therefore answer different questions.

**Inspection question:** Are present timestamps written with the documented text shape?

**Documentation:** [Series.str.fullmatch()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.fullmatch.html).

**Shape versus calendar example:** `2001-99-19 00:00:00` has four year digits, two month digits
and the expected separators, so it fits the text shape. It still fails calendar conversion.
`2001/07/19 00:00:00` has different separators and fails this documented shape check.


In [ ]:
# Check the written timestamp shape separately from whether its digits describe a possible
# calendar date.

# Four digits, a hyphen, two digits, another hyphen, two digits, a space, then time digits
# and colons describe the expected layout. For example, 2001-99-19 00:00:00 has that layout
# even though month 99 is impossible.
timestamp_pattern = r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"
for column in ["discovery_date", "contained_date"]:
    date_text = raw[column].astype("string")

    # fullmatch requires the entire value to fit the pattern, so extra characters at either
    # end can be flagged. Missing inputs receive False here only to make selection possible;
    # the presence condition below keeps them untested.
    correct_shape = date_text.str.fullmatch(timestamp_pattern, na=False)

    # ~ reverses the shape decision. Combining "present" AND "not correct shape" flags a
    # written-format difference without counting absent dates as bad strings.
    wrong_shape = date_text.notna() & ~correct_shape
    print(column, "present values with unexpected shapes:", wrong_shape.sum())
    # Inspect original examples because a wrong separator or extra space is a different issue
    # from an impossible calendar value. Neither screen replaces the other.
    display(raw.loc[wrong_shape, [column]].head(5))


## 9. Worked log entry: surrounding whitespace in names — T01

Return to the name-formatting finding and prepare a complete record of it. Our question is:
**Can we group records consistently by their recorded fire names?** The original and trimmed names
should appear side by side in the evidence. The worked severity is **Medium** because name grouping can be affected
while other fields remain usable. A shared name still does not establish shared incident identity.

| Object we will build | What it contains | Meaning of one row |
| --- | --- | --- |
| `evidence_t01` | Original values supporting T01, plus reference labels | One affected source record |
| `log_entry` | A Python dictionary containing the answers for one issue | Not a table yet: field names paired with answers |
| `issue_log` | The dictionary converted into a DataFrame | One issue, however many records support it |

`source_row` preserves the original zero-based pandas row label from this unchanged import. It is not an incident ID
or a physical CSV line number. `issue_id` connects the evidence to its issue-log entry.

Some log fields are measured by code; others require your judgement. State the question before choosing severity.

| Severity | Meaning for the stated analysis |
| --- | --- |
| Low | Limited effect on the stated question. |
| Medium | A meaningful part of the comparison or grouping may be affected. |
| High | A central result cannot be supported reliably without a decision. |
| Not assigned | More investigation is needed before judging the effect. |

A percentage alone does not select a severity. A **proposed action** is a plan; **status** records progress;
**verification** explains how the later decision or treatment will be checked.


### 9.1 Identify and count affected names

Repeat the short rule here so the worked log can be followed as a complete process. Only present names can be inspected for surrounding whitespace.

**Inspection question:** How many of the names we could actually check change when trimmed?

**Documentation:** [Series.str.strip()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html) · [Series.notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [float()](https://docs.python.org/3.14/library/functions.html#float).

**The complete whitespace rule:** A record is flagged when its name is **present AND different
after trimming**. For example, if a 100-record file has 50 present names and 10 of those names
change, the checked population is 50 and the affected percentage is 20%. The 50 absent names
remain outside this spelling comparison; they did not pass it.


In [ ]:
# The T01 log needs the full affected count and the population whose names could actually be
# inspected.

# Recreate the original and trimmed Series here so the evidence-to-log sequence is
# self-contained. The comparison asks whether whitespace at the ends changes a present name.
names = raw["fire_name"].astype("string")
trimmed_names = names.str.strip()
present_names = names.notna()
changed_by_trimming = names != trimmed_names
# Both source presence and a changed comparison are required. Filling unknown flags with
# False excludes unavailable spellings rather than treating them as inspected names.
name_has_spaces = (present_names & changed_by_trimming).fillna(False)

space_count = name_has_spaces.sum()
# The denominator counts present names, not all records. If 10 of 50 present names change in
# a 100-row file, this check reports 20%, not 10%.
names_checked = present_names.sum()

# Start with an unavailable percentage because an empty eligible population cannot support
# division or a pass. The if branch calculates it only when at least one present name was
# inspected.
space_percent = float("nan")
if names_checked > 0:
    space_percent = space_count / names_checked * 100
else:
    print("Untested: there are no present names to inspect.")
print("Affected names:", space_count)
print("Present names checked:", names_checked)
# Four displayed decimals preserve small proportions, while the full affected and checked
# counts above make the percentage reproducible.
print(f"Affected percentage: {space_percent:.4f}%")


### 9.2 Build evidence with the original and comparison values

Keep every affected source record with its name and state. Add the trimmed value and the number of characters removed so the evidence explains the difference.

**Inspection question:** Can a reviewer see the exact whitespace difference and locate its source record?

**Documentation:** [DataFrame.copy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [Series.loc](https://pandas.pydata.org/docs/reference/api/pandas.Series.loc.html) · [Series.str.len()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.len.html) · [DataFrame.head()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html).


In [ ]:
# Evidence must demonstrate the whitespace difference, retain every affected record and
# preserve a route back to its original values.

evidence_columns = ["fire_name", "state"]
# Select flagged source records into a copy before adding comparison information. fire_name
# stays original in this complete table; state supplies context.
evidence_t01 = raw.loc[name_has_spaces, evidence_columns].copy()

# Save the original label as a normal column so index-free exports still identify
# raw.loc[source_row]. T01 then connects every supporting record to one issue.
evidence_t01["source_row"] = evidence_t01.index
evidence_t01["issue_id"] = "T01"

# Retrieve the comparison at the evidence row labels, not at the first few positions in the
# Series. This keeps original and trimmed names from the same record together.
evidence_t01["trimmed_name"] = trimmed_names.loc[evidence_t01.index]

# The original length minus the trimmed length counts removed end characters. " Oak " has
# five characters and "Oak" has three, so the evidence shows 2 removed characters.
original_lengths = names.loc[evidence_t01.index].str.len()
trimmed_lengths = evidence_t01["trimmed_name"].str.len()
evidence_t01["characters_removed"] = original_lengths - trimmed_lengths

# Put references, original, comparison and measured difference next to one another so a
# reviewer can reconstruct the reasoning.
evidence_t01 = evidence_t01[[
    "issue_id", "source_row", "fire_name", "trimmed_name", "characters_removed", "state"
]]
pd.set_option("display.max_columns", None)
print("Rows in full T01 evidence:", len(evidence_t01))

# Use a display-only copy to bracket names: [ Oak ] makes outside spaces visible. Export
# evidence_t01, because the preview contains only ten rows and adds presentation characters
# not present in the source.
name_preview = evidence_t01.head(10).copy()
name_preview["fire_name"] = "[" + names.loc[name_preview.index] + "]"
name_preview["trimmed_name"] = "[" + name_preview["trimmed_name"] + "]"
display(name_preview)


### 9.3 Record the measured facts

The denominator is the number of present names, not the whole dataset. The reference finding is 77,704 affected names out of 919,986 present names (about 8.4462%).

**Inspection question:** What did we find, and how many records did we actually check?

**Documentation:** [dict](https://docs.python.org/3.14/library/stdtypes.html#dict) · [int()](https://docs.python.org/3.14/library/functions.html#int) · [round()](https://docs.python.org/3.14/library/functions.html#round).


In [ ]:
# Separate measured facts from written judgement so the count can be checked independently of
# our interpretation.

# A dictionary keeps named answers for ONE issue. Converting it to a table later gives one
# summary row even when thousands of source rows support that finding.
log_entry = {

    # Use T01 consistently in the log and evidence so the summary points to the correct
    # supporting records.
    "issue_id": "T01",

    # Describe what the check establishes. An absence or a disagreement is observable;
    # calling a source value an error would require additional evidence.
    "finding_type": "Text-format difference",

    "field": "fire_name",

    # space_count counts the full flagged population. int stores a plain whole-number count;
    # the length of a display preview is not an alternative count.
    "affected_rows": int(space_count),

    # names_checked is the population that could actually be tested. Excluding untestable
    # records from this denominator prevents us from making the issue appear smaller.
    "checked_rows": int(names_checked),

    # Keep four decimal places because rounding a very small non-zero result to two could
    # display 0.00%. An unavailable percentage remains unavailable; it is not evidence of a
    # pass.
    "percent": round(space_percent, 4)
}


### 9.4 Connect the facts to the analysis and evidence

The counts describe the observation. We now explain why we checked it and where a reviewer can find the supporting records.

**Inspection question:** Could another student reproduce this finding and locate its evidence?

**Documentation:** [dict](https://docs.python.org/3.14/library/stdtypes.html#dict).


In [ ]:
# Add the reason for the check and a route back to its evidence; counts alone do not explain
# why the finding matters.

# The question determines relevance. Missing dates matter to duration; a date-year
# disagreement matters to annual grouping. A different question may lead to a different
# priority.
log_entry["question"] = 'Can we group records consistently by their recorded fire names?'

# State the condition that would make the question answerable. This connects an acceptance or
# comparison rule to its purpose rather than treating it as an arbitrary instruction.
log_entry["expectation"] = (
    'Surrounding whitespace should not create separate name groups for this comparison.'
)

# A notebook section and a reproducible condition let another person rerun the check. The
# issue label by itself would not explain how we arrived at the count.
log_entry["check_reference"] = 'Notebook 02, section 9.1: present names differ from trimmed_names.'

# This answer stores the name evidence_t01 as text; the actual DataFrame remains a separate
# object. The name tells the reviewer which table to consult, but does not save that table.
log_entry["evidence_table"] = "evidence_t01"

# Say whether every affected row was retained. Without that distinction, a ten-row example
# could be mistaken for the full evidence population.
log_entry["evidence_scope"] = (
    'All affected records are included; head(10) and name_preview show ten rows only.'
)


### 9.5 Explain the effect and propose the next step

We have evidence, but inspection alone does not settle how the data should be treated. The following wording is a worked judgement for the stated question.

**Inspection question:** What could this finding change in our analysis, and what should happen next?


In [ ]:
# Write a justified decision proposal: the detection code supplies observations, while these
# answers explain their analytical consequences.

# Explain the consequence: different name spellings can split one intended label group even
# though both strings are present.
log_entry["impact"] = (
    'Names that differ only in surrounding whitespace may be counted in separate '
    'groups.'
)

# Priority follows the stated question, not a percentage cut-off. The explanation below must
# make the chosen priority defensible for this particular use.
log_entry["severity"] = 'Medium'

# This is worked wording for the supplied reference finding. Compare it with your computed
# result and analysis population before reusing it.
log_entry["severity_reason"] = (
    'Name-based grouping can be affected, while other information such as size and '
    'cause is still usable.'
)

# Keep the decision as text at the inspection stage. Investigating recovery, defining a
# subset or agreeing a comparison policy is a proposal; assigning this answer changes no
# source values.
log_entry["proposed_action"] = (
    'Review the evidence and agree whether surrounding whitespace should be removed in '
    'a separate cleaned dataset.'
)

# Evidence exists, but the treatment has not been agreed or verified. Keeping the issue open
# prevents a proposed action being read as a completed correction.
log_entry["status"] = "Open - decision required"

# Name the later checks that could show the decision worked. Repeating the relevant check and
# reconciling counts is more convincing than merely writing "fixed".
log_entry["verification"] = (
    'After agreed cleaning, repeat the whitespace check, explain any change in '
    'distinct-name counts and confirm that the row count stayed the same.'
)


### 9.6 Display, verify and optionally save the entry

The completed dictionary will become one row in `issue_log`. Keep that summary with the full `evidence_t01` evidence table. Each notebook creates its own one-entry log; they are not combined automatically. To add it to your shared log, match fields by meaning and update an existing T01 entry instead of duplicating it.

**Inspection question:** Does the recorded affected count agree with the evidence we kept?

**Documentation:** [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [pd.set_option()](https://pandas.pydata.org/docs/reference/api/pandas.set_option.html) · [DataFrame.T](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.T.html) · [DataFrame.to_csv()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) · [len()](https://docs.python.org/3.14/library/functions.html#len).


In [ ]:
# Show the one-issue summary and compare its affected count with the complete supporting
# evidence before exporting.

# The outer list contains one dictionary, so Pandas interprets its keys as headings and its
# answers as one row. Re-running recreates this table; it does not append a second issue.
issue_log = pd.DataFrame([log_entry])

# Long written answers need to stay visible for review. Transposing the displayed table puts
# field names down the page; the stored issue_log still has one issue row.
pd.set_option("display.max_colwidth", None)
display(issue_log.T)

# These two counts should agree because one evidence row represents one flagged source
# record. A difference would suggest the evidence filter or the logged count came from a
# different check.
print("Rows in full evidence:", len(evidence_t01))
print("Affected rows recorded in log:", log_entry["affected_rows"])

# OPTIONAL: uncomment the next two commands to save both complete tables. index=False avoids
# a redundant index column; source_row still preserves traceability. Saving again replaces
# these CSVs, so retain reusable wording in the notebook.
# issue_log.to_csv("Issue_Log_T01.csv", index=False)
# evidence_t01.to_csv("evidence_t01.csv", index=False)


### Practise the same process: an undefined cause — C01

A cell containing `Missing/Undefined` is filled in, but it does not tell us the cause of the fire.
That is different from an empty cell. Keep the T01 entry before creating a new `log_entry` dictionary.

1. State a question such as “What proportion of all reported records has each cause?”
2. Reuse `undefined_cause` from section 3.1 to select records labelled `Missing/Undefined`.
3. Create `evidence_c01` with `fire_name`, `state` and `fire_cause`, then add `source_row` and `issue_id` set to `C01`.
4. For this all-record question, record **166,723 affected records out of 1,880,465 checked records**.
   A ten-row preview is not the full count.
5. Explain the impact in one sentence, for example: “These records cannot be assigned to a known cause from the available label.”
6. Write one C01 log entry. Point it to `evidence_c01`, explain your severity, suggest a next step and keep the status Open.

Do not guess a cause from the label. If a later analysis uses only records with known causes,
explain that smaller analysis population separately. An allowed category can still limit what we know.


## 10. Carry the findings forward

| ID / check | Observation | Next decision |
| --- | --- | --- |
| C01 | 166,723 undefined cause labels | Keep them visible; define the population for cause analysis. |
| T01 | 77,704 names with surrounding spaces | Use the worked evidence and log above to agree a policy. |
| T02 | 3,328 candidate placeholder names | Investigate meaning before treating them as missing. |
| R02 | 43 newly flagged members after text comparison | Investigate identity; do not assume they are duplicate fires. |
| T03 | 22 non-ASCII name records | Separate legitimate characters from possible encoding problems. |
| P01 | No numeric conversion failures or infinities | Continue to range checks. |
| P02 | No shape or parsing failures among present dates | Continue to date relationships; missing dates remain untested. |

For a new dataset, replace the column selections and format expectations with those from its own
documentation. Do not turn the observed values into an “allowed” list: that would force a pass.


### Study check and next investigation

Explain the difference between a missing cell, an undefined label and a possible placeholder.
Then explain why the name-version check removes repeated **pairs** before flagging repeated **comparison values**.

**Next:** Notebook 03 uses interpretable values to check allowed categories, numeric ranges, date relationships,
coverage and unusual measurements. Conversion success is the starting point for those checks, not the final verdict.
